# Stage 3: Data Preprocessing & Feature Engineering
## AI-Based Student Dropout Prediction and Early Warning System

This notebook demonstrates the complete data preprocessing workflow:
1. Loading the clean UCI Student dataset.
2. Quality checks (missing values, duplicates, dimensions).
3. Binary mapping of target variable: `Dropout_Risk` (1 for Dropout, 0 for Graduate/Enrolled).
4. Separating features ($X$) and target ($y$) while preventing target leakage.
5. Categorizing features into Numerical, Binary, and Categorical.
6. Addressing the limitation of 2nd-semester features in an Early Warning context.
7. Stratified 80:20 Train-Test split with `random_state=42`.
8. Feature scaling using `StandardScaler` fitted strictly on training data.
9. Exporting processed train/test splits and fitted preprocessors.

In [ ]:
import os
import json
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

# Locate dataset file
dataset_path = os.path.join("..", "dataset", "student_data_clean.csv")
df = pd.read_csv(dataset_path)
print(f"Dataset loaded successfully. Shape: {df.shape}")
df.head()

### Step 1: Target Transformation to Binary Risk
- `Dropout` = **1** (Student at risk of dropping out)
- `Graduate` = **0** (Student successfully completed)
- `Enrolled` = **0** (Student currently active/retained)

We keep the original `Target` column in the dataframe, but create a new binary `Dropout_Risk` column.

In [ ]:
df['Dropout_Risk'] = (df['Target'] == 'Dropout').astype(int)
print("Binary target distribution:")
display(df['Dropout_Risk'].value_counts())
print("\nPercentage distribution:")
display(df['Dropout_Risk'].value_counts(normalize=True) * 100)

### Step 2: Separate Features (X) and Target (y)
To avoid **Target Leakage**, we must drop both `Target` and `Dropout_Risk` from the input feature matrix $X$.

In [ ]:
X = df.drop(columns=['Target', 'Dropout_Risk'])
y = df['Dropout_Risk']
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

### Step 3: Stratified 80:20 Train-Test Split
- 80% Training set (3,539 students)
- 20% Testing set (885 students)
- `stratify=y` ensures identical 32.1% at-risk ratio in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train set: {X_train.shape[0]} rows | At-risk ratio: {y_train.mean():.4f}")
print(f"Test set:  {X_test.shape[0]} rows | At-risk ratio: {y_test.mean():.4f}")

### Step 4: Preprocessing Pipeline (Preventing Data Leakage)
`StandardScaler` is fitted **ONLY** on `X_train`. We then use `.transform()` on `X_test`.

In [ ]:
continuous_cols = [
    "Previous qualification (grade)", "Admission grade", "Age at enrollment",
    "Curricular units 1st sem (credited)", "Curricular units 1st sem (enrolled)",
    "Curricular units 1st sem (evaluations)", "Curricular units 1st sem (approved)",
    "Curricular units 1st sem (grade)", "Curricular units 1st sem (without evaluations)",
    "Curricular units 2nd sem (credited)", "Curricular units 2nd sem (enrolled)",
    "Curricular units 2nd sem (evaluations)", "Curricular units 2nd sem (approved)",
    "Curricular units 2nd sem (grade)", "Curricular units 2nd sem (without evaluations)",
    "Unemployment rate", "Inflation rate", "GDP"
]
passthrough_cols = [c for c in X.columns if c not in continuous_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ("num_scaler", StandardScaler(), continuous_cols),
        ("pass", "passthrough", passthrough_cols)
    ]
)

preprocessor.fit(X_train)
print("Preprocessor fitted strictly on X_train. Ready for modeling!")